# 08 — All results: one set of Markdown tables

Verify accepted frontal, central/temporal, combined and historical parent artifacts.
Aggregate saved correlations with channel/window/record balancing. No EEG array
is loaded, no denoising/training is rerun and no predictions change. Native
EEG has no reconstruction SNR. This notebook exports ALL_RESULTS_TABLES.md.


In [ ]:
RUN_SPEC = {'run_id': 'all-results-tables-001', 'campaign_id': 'results-tables-20261017', 'kernel': 'satyapaladugu/eog-all-results-tables', 'git_sha': 'RESOLVED_TO_PUBLISHED_SHA_AT_SUBMISSION', 'git_url': 'https://github.com/saltypal/Lateral-eye-Artifact-removal.git', 'created_utc': '2026-10-17T10:03:07.744116+00:00', 'output': '/kaggle/working/results/results-tables-20261017/all-results-tables-001'}
import os,sys,pathlib,tempfile,subprocess,json,platform
assert pathlib.Path('/kaggle/input').exists()
print('BOOTSTRAP_START',RUN_SPEC['run_id'],RUN_SPEC['git_sha'],flush=True)
CHECKOUT=pathlib.Path(tempfile.mkdtemp(prefix='results-tables-source-'))
subprocess.run(['git','clone','--quiet',RUN_SPEC['git_url'],str(CHECKOUT)],check=True)
subprocess.run(['git','-C',str(CHECKOUT),'checkout','--quiet',RUN_SPEC['git_sha']],check=True)
actual=subprocess.check_output(['git','-C',str(CHECKOUT),'rev-parse','HEAD'],text=True).strip()
assert actual==RUN_SPEC['git_sha']
subprocess.run([sys.executable,'-m','pip','install','-q','pytest==8.3.5','tabulate==0.9.0'],check=True,timeout=120)
sys.path.insert(0,str(CHECKOUT/'src'))
from eog_clean.data import save_json
import pandas,numpy,scipy
OUTPUT=pathlib.Path(RUN_SPEC['output']);OUTPUT.mkdir(parents=True,exist_ok=True)
save_json(OUTPUT/'run_spec.json',RUN_SPEC)
save_json(OUTPUT/'environment.json',{'source_sha':actual,'python':sys.version,'platform':platform.platform(),
    'pandas':pandas.__version__,'numpy':numpy.__version__,'scipy':scipy.__version__,'hardware':'Kaggle CPU'})


## Verify aggregation, then create every table

In [ ]:
environment=os.environ.copy();environment['PYTHONPATH']=str(CHECKOUT/'src')
test=subprocess.run([sys.executable,'-m','pytest','-q',str(CHECKOUT/'tests/test_results_tables.py'),
    '--junitxml='+str(OUTPUT/'table_tests.xml')],capture_output=True,text=True,env=environment)
(OUTPUT/'table_tests.txt').write_text(test.stdout+'\n'+test.stderr)
print(test.stdout,flush=True)
if test.returncode:raise RuntimeError('Saved-correlation aggregation fixtures failed')
from eog_clean.results_tables import build
build('/kaggle/input',OUTPUT)


## Consolidated result document

In [ ]:
from IPython.display import display,Markdown
display(Markdown((OUTPUT/'ALL_RESULTS_TABLES.md').read_text()))
print((OUTPUT/'completion.json').read_text())
save_json(OUTPUT/'execution_state.json',{'state':'complete','source_sha':actual,
    'numerical_scope':'aggregation of saved metrics only','predictions_changed':False})
